In [2]:
import pandas as pd
import glob
import os

# 1) 최종 파일 읽기
final_path = "amore_final.csv"
final_df = pd.read_csv(final_path)

# 2) 브랜드별 CSV들이 있는 디렉터리 (스크린샷 기준)
BRAND_DIR = "../data_csv/amore_csv"   # <- 여기 중요

brand_files = glob.glob(os.path.join(BRAND_DIR, "*.csv"))

print("브랜드 CSV 개수:", len(brand_files))

# 3) (상품명, brand) 매핑 테이블 만들기
brand_frames = []

for file in brand_files:
    brand = os.path.splitext(os.path.basename(file))[0]  # 파일명에서 .csv 제거 = 브랜드명
    df_b = pd.read_csv(file)

    if "상품명" not in df_b.columns:
        continue

    tmp = df_b[["상품명"]].copy()
    tmp["brand"] = brand
    brand_frames.append(tmp)

brand_map = pd.concat(brand_frames, ignore_index=True).drop_duplicates()

print("매핑용 상품 수:", len(brand_map))

# 4) amore_final.csv 에 brand 붙이기
final_df = final_df.merge(brand_map, on="상품명", how="left")

# 5) product_id = brand_상품명
final_df["product_id"] = final_df.apply(
    lambda row: f"{row['brand']}_{row['상품명']}" if pd.notna(row["brand"]) else None,
    axis=1
)

# 6) 저장 (덮어쓰기)
final_df.to_csv(final_path, index=False, encoding="utf-8-sig")

print(">> brand, product_id 채움 완료")
print(final_df[["상품명", "brand", "product_id"]].head())

브랜드 CSV 개수: 0


ValueError: No objects to concatenate

In [6]:
# 12_10_fill_brand_from_url.py

import pandas as pd
import glob
import os

# 1) 경로 설정 (네가 말한 그대로)
BRAND_DIR = "/Users/Mac/Desktop/project1/STUDY-DATA/first_week/data_csv/amore_csv"
FINAL_CSV = "/Users/Mac/Desktop/project1/STUDY-DATA/second_week/amore_final.csv"

# 2) amore_final.csv 로드
df_final = pd.read_csv(FINAL_CSV)
df_final = df_final.drop(columns=["brand", "product_id"], errors="ignore")

if "URL" not in df_final.columns:
    raise RuntimeError("amore_final.csv 에 'URL' 컬럼이 없음")

# 3) 브랜드 CSV들에서 (URL, brand) 매핑 만들기
brand_files = glob.glob(os.path.join(BRAND_DIR, "*.csv"))
print("브랜드 CSV 파일 수:", len(brand_files))

if len(brand_files) == 0:
    raise RuntimeError("BRAND_DIR 아래에서 CSV 파일을 하나도 못 찾음")

brand_rows = []

for path in brand_files:
    brand_name = os.path.splitext(os.path.basename(path))[0]  # 파일명 = 브랜드명
    df_b = pd.read_csv(path)

    # URL 후보 컬럼들
    url_candidates = [c for c in ["URL", "url", "Url", "상품URL", "링크"] if c in df_b.columns]
    if not url_candidates:
        # 이 브랜드 CSV에 URL이 없으면 스킵
        continue

    url_col = url_candidates[0]

    tmp = df_b[[url_col]].copy()
    tmp = tmp.rename(columns={url_col: "URL"})
    tmp["brand"] = brand_name
    brand_rows.append(tmp)

if not brand_rows:
    raise RuntimeError("브랜드 CSV들에서 URL 컬럼을 가진 파일을 하나도 못 찾음")

brand_map = pd.concat(brand_rows, ignore_index=True).dropna(subset=["URL"]).drop_duplicates()

print("매핑용 URL 개수:", len(brand_map))

# 4) URL 기준 merge → brand 붙이기
df_final = df_final.merge(brand_map, on="URL", how="left")

# 5) product_id 생성 (brand_상품명)
def make_product_id(row):
    b = row["brand"]
    name = row["상품명"]
    if isinstance(b, str) and b != "":
        return f"{b}_{name}"
    return None

df_final["product_id"] = df_final.apply(make_product_id, axis=1)

# 6) 저장 (amore_final.csv 덮어쓰기)
df_final.to_csv(FINAL_CSV, index=False, encoding="utf-8-sig")

print("\n✅ URL 기반 brand / product_id 채워넣기 완료")
print(df_final[["상품명", "URL", "brand", "product_id"]].head())

브랜드 CSV 파일 수: 30
매핑용 URL 개수: 1501

✅ URL 기반 brand / product_id 채워넣기 완료
              상품명                                                URL brand  \
0    메이크온 시너지 마스크  https://www.amoremall.com/kr/ko/product/detail...  메이크온   
1   가볍게 마시는 히알루론산  https://www.amoremall.com/kr/ko/product/detail...  메이크온   
2  스킨 라이트 테라피 III  https://www.amoremall.com/kr/ko/product/detail...  메이크온   
3   스킨 라이트 테라피 3S  https://www.amoremall.com/kr/ko/product/detail...  메이크온   
4    젬 소노 테라피 릴리프  https://www.amoremall.com/kr/ko/product/detail...  메이크온   

            product_id  
0    메이크온_메이크온 시너지 마스크  
1   메이크온_가볍게 마시는 히알루론산  
2  메이크온_스킨 라이트 테라피 III  
3   메이크온_스킨 라이트 테라피 3S  
4    메이크온_젬 소노 테라피 릴리프  


In [7]:
# 12_10_fill_brand_from_url.py

import pandas as pd
import glob
import os

# 1) 경로 설정 (네가 말한 그대로)
BRAND_DIR = "/Users/Mac/Desktop/project1/STUDY-DATA/first_week/data_csv/amore_csv"
FINAL_CSV = "/Users/Mac/Desktop/project1/STUDY-DATA/second_week/amore_with_category.csv"

# 2) amore_final.csv 로드
df_final = pd.read_csv(FINAL_CSV)
df_final = df_final.drop(columns=["brand", "product_id"], errors="ignore")

if "URL" not in df_final.columns:
    raise RuntimeError("amore_final.csv 에 'URL' 컬럼이 없음")

# 3) 브랜드 CSV들에서 (URL, brand) 매핑 만들기
brand_files = glob.glob(os.path.join(BRAND_DIR, "*.csv"))
print("브랜드 CSV 파일 수:", len(brand_files))

if len(brand_files) == 0:
    raise RuntimeError("BRAND_DIR 아래에서 CSV 파일을 하나도 못 찾음")

brand_rows = []

for path in brand_files:
    brand_name = os.path.splitext(os.path.basename(path))[0]  # 파일명 = 브랜드명
    df_b = pd.read_csv(path)

    # URL 후보 컬럼들
    url_candidates = [c for c in ["URL", "url", "Url", "상품URL", "링크"] if c in df_b.columns]
    if not url_candidates:
        # 이 브랜드 CSV에 URL이 없으면 스킵
        continue

    url_col = url_candidates[0]

    tmp = df_b[[url_col]].copy()
    tmp = tmp.rename(columns={url_col: "URL"})
    tmp["brand"] = brand_name
    brand_rows.append(tmp)

if not brand_rows:
    raise RuntimeError("브랜드 CSV들에서 URL 컬럼을 가진 파일을 하나도 못 찾음")

brand_map = pd.concat(brand_rows, ignore_index=True).dropna(subset=["URL"]).drop_duplicates()

print("매핑용 URL 개수:", len(brand_map))

# 4) URL 기준 merge → brand 붙이기
df_final = df_final.merge(brand_map, on="URL", how="left")


# 6) 저장 (amore_final.csv 덮어쓰기)
df_final.to_csv(FINAL_CSV, index=False, encoding="utf-8-sig")

print("\n✅ product_id 채워넣기 완료")
print(df_final[["상품명", "URL", "brand"]].head())

브랜드 CSV 파일 수: 30
매핑용 URL 개수: 1501

✅ product_id 채워넣기 완료
              상품명                                                URL brand
0    메이크온 시너지 마스크  https://www.amoremall.com/kr/ko/product/detail...  메이크온
1   가볍게 마시는 히알루론산  https://www.amoremall.com/kr/ko/product/detail...  메이크온
2  스킨 라이트 테라피 III  https://www.amoremall.com/kr/ko/product/detail...  메이크온
3   스킨 라이트 테라피 3S  https://www.amoremall.com/kr/ko/product/detail...  메이크온
4    젬 소노 테라피 릴리프  https://www.amoremall.com/kr/ko/product/detail...  메이크온
